# Structured Output vs Raw JSON 파싱 비교 테스트

OpenRouter 프록시를 통해 `gpt-oss-120b`를 사용할 때,
`with_structured_output()`이 실패하는 이유와 현재 raw JSON 방식의 동작을 비교합니다.

| 방법 | 방식 | OpenRouter 호환 | 스트리밍 | 비고 |
|------|------|:---:|:---:|------|
| A | `with_structured_output()` | ❌ | ❌ | `response_format.json_schema` 전달 실패 |
| B | Raw JSON + regex + json_repair | ✅ | ✅ | **현재 사용 중** |
| C | `response_format={"type": "json_object"}` | △ | ❌ | 일부 모델만 지원 |

## 0. 공통 설정

In [ ]:
import json
import os
import re
from typing import Literal

from dotenv import load_dotenv
from json_repair import repair_json
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

load_dotenv(override=True)

In [ ]:
# ── Pydantic 스키마 (supervisor.py RouteResponse 간소화 버전) ──

class RouteResponse(BaseModel):
    """Supervisor의 라우팅 결정"""
    next: Literal["yield_agent", "wads_agent", "map_agent", "fail_history_agent", "FINISH"] = Field(
        description="다음에 실행할 에이전트"
    )
    lotcd: str = Field(default="", description="3~4자리 제품코드 (예: 4SS, 5NA)")
    ref_date: str = Field(default="", description="기준날짜 YYYYMMDD")
    filter_params: list[str] = Field(default=[], description="파라미터 필터 (예: ['VTH'])")
    unit: str = Field(default="weekly", description='"weekly" | "monthly" | "daily"')
    periods: int = Field(default=0, description="조회 기간 수")
    message: str = Field(default="", description="사용자에게 전달할 한국어 메시지")

# 스키마 확인
print(json.dumps(RouteResponse.model_json_schema(), indent=2, ensure_ascii=False))

In [ ]:
# ── LLM 초기화 ──

llm = ChatOpenAI(
    model=os.getenv("DEFAULT_MODEL", "gpt-oss-120b"),
    base_url=os.getenv("OPENROUTER_BASE_URL"),
    api_key=os.getenv("OPENROUTER_API_KEY"),
    temperature=0,
)

print(f"모델: {llm.model_name}")
print(f"Base URL: {llm.openai_api_base}")

In [ ]:
# ── 테스트 입력 ──

SYSTEM_PROMPT = """You are a routing supervisor.
Respond with a JSON object matching this schema:
{
  "next": "yield_agent" | "wads_agent" | "map_agent" | "fail_history_agent" | "FINISH",
  "lotcd": "<product code or empty>",
  "ref_date": "<YYYYMMDD or empty>",
  "filter_params": [],
  "unit": "weekly",
  "periods": 0,
  "message": "<Korean message>"
}
Output ONLY the JSON. No markdown, no explanation."""

TEST_QUERY = "오늘 4SS 수율 VTH만 보여줘"
print(f"테스트 쿼리: {TEST_QUERY}")

---
## 방법 A: `with_structured_output()` — OpenRouter에서 실패하는 방식

LangChain이 내부적으로 OpenAI API에 `response_format` 파라미터를 보냅니다:

```
POST https://<openrouter-base>/v1/chat/completions
{
    "model": "gpt-oss-120b",
    "messages": [...],
    "response_format": {           ← OpenRouter가 이걸 전달 못함
        "type": "json_schema",
        "json_schema": {
            "name": "RouteResponse",
            "strict": true,
            "schema": { ... }
        }
    }
}
```

**실패 시나리오:**
1. OpenRouter가 `response_format`을 무시하거나 변환 실패
2. LLM이 자유 형식 텍스트를 반환
3. LangChain이 Pydantic 파싱 시도 → `ValidationError`
4. supervisor의 except절에서 fallback → "요청을 이해하지 못했습니다"

In [ ]:
# 방법 A 테스트: with_structured_output()

structured_llm = llm.with_structured_output(RouteResponse)

try:
    result = structured_llm.invoke([
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": TEST_QUERY},
    ])
    print(f"✅ 성공!")
    print(f"  next          = {result.next}")
    print(f"  lotcd         = {result.lotcd}")
    print(f"  ref_date      = {result.ref_date}")
    print(f"  filter_params = {result.filter_params}")
    print(f"  message       = {result.message}")
except Exception as e:
    print(f"❌ 실패: {type(e).__name__}")
    print(f"   {e}")
    print()
    print("   → 이것이 supervisor.py 주석에서 말하는 버그입니다.")
    print("   → OpenRouter가 response_format/function_calling을")
    print("     gpt-oss-120b에 제대로 전달하지 못해서 발생합니다.")

---
## 방법 B: Raw JSON + regex + json_repair — 현재 사용 중

API에 아무 특수 파라미터 없이 일반 chat completion으로 호출합니다:

```
POST https://<openrouter-base>/v1/chat/completions
{
    "model": "gpt-oss-120b",
    "messages": [
        {"role": "system", "content": "...JSON schema... Output ONLY the JSON."},
        {"role": "user", "content": "오늘 4SS 수율 VTH만 보여줘"}
    ],
    "max_tokens": 2048
    // response_format 없음! → 어떤 프록시든 동작
}
```

LLM 응답 → `<think>` 분리 → regex JSON 추출 → `json_repair` → Pydantic 검증

### Step 1: LLM 스트리밍 호출 + `<think>` 태그 실시간 분리

In [ ]:
# Step 1: 스트리밍 호출 — supervisor_node가 실제로 하는 방식

raw_text = ""
thinking_buf = ""
in_think = False

print("[스트리밍 시작]")
for chunk in llm.stream(
    [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": TEST_QUERY},
    ],
    max_tokens=2048,
):
    token = chunk.content or ""
    if not token:
        continue
    raw_text += token

    # <think> 태그 실시간 파싱 (supervisor_node 동일 로직)
    if not in_think and "<think>" in raw_text and "</think>" not in raw_text:
        in_think = True
        thinking_buf = raw_text.split("<think>", 1)[1]
        print(f"  💭 [thinking 시작] {thinking_buf}", end="")
        continue

    if in_think:
        if "</think>" in raw_text:
            in_think = False
            think_content = raw_text.split("<think>", 1)[1].split("</think>", 1)[0]
            remaining = think_content[len(thinking_buf):]
            if remaining:
                print(remaining)
            print("  💭 [thinking 종료]")
        else:
            current_think = raw_text.split("<think>", 1)[1]
            new_part = current_think[len(thinking_buf):]
            if new_part:
                print(new_part, end="")
                thinking_buf = current_think

raw_text = raw_text.strip()
print(f"\n[LLM raw 응답 전체]\n{raw_text}")

### Step 2: `<think>` 태그 제거

In [ ]:
# Step 2: <think> 태그 제거
# 닫는 태그가 없는 경우도 처리: <think>...끝 → 전부 제거

clean_text = re.sub(r"<think>.*?</think>", "", raw_text, flags=re.DOTALL).strip()
clean_text = re.sub(r"<think>.*", "", clean_text, flags=re.DOTALL).strip()

print(f"[think 제거 후]\n{clean_text}")

### Step 3: 정규식으로 JSON 블록 추출

In [ ]:
# Step 3: JSON 블록 추출
# 우선순위: ```json ... ``` > { ... } (clean_text) > { ... } (raw_text)

json_match = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", clean_text, re.DOTALL)
if json_match:
    print("📌 마크다운 코드블록에서 추출")
else:
    json_match = re.search(r"(\{.*\})", clean_text, re.DOTALL)
    if json_match:
        print("📌 clean_text에서 { ... } 추출")
    else:
        json_match = re.search(r"(\{.*\})", raw_text, re.DOTALL)
        if json_match:
            print("📌 raw_text에서 { ... } 추출 (think 안에 JSON이 있었던 케이스)")

if json_match:
    json_str = json_match.group(1)
    print(f"\n[추출된 JSON]\n{json_str}")
else:
    print("❌ JSON을 찾을 수 없습니다")
    json_str = None

### Step 4: `json_repair`로 malformed JSON 복구

In [ ]:
# Step 4: json_repair → json.loads

if json_str:
    repaired = repair_json(json_str)
    data = json.loads(repaired)
    
    print("[파싱된 dict]")
    print(json.dumps(data, ensure_ascii=False, indent=2))
else:
    data = None
    print("⏭️ JSON이 없으므로 스킵")

### Step 5: Pydantic 검증 → `RouteResponse`

In [ ]:
# Step 5: Pydantic 검증

if data:
    try:
        decision = RouteResponse(**data)
        print("✅ Pydantic 검증 성공!\n")
        print(f"  next          = {decision.next}")
        print(f"  lotcd         = {decision.lotcd}")
        print(f"  ref_date      = {decision.ref_date}")
        print(f"  filter_params = {decision.filter_params}")
        print(f"  unit          = {decision.unit}")
        print(f"  periods       = {decision.periods}")
        print(f"  message       = {decision.message}")
    except Exception as e:
        print(f"❌ Pydantic 검증 실패: {e}")
        print("   → supervisor에서는 여기서 fallback RouteResponse(next='FINISH', ...) 반환")
else:
    print("⏭️ data가 없으므로 스킵")

---
## 방법 C: `response_format={"type": "json_object"}` — 부분 지원

`json_schema`가 아닌 `json_object`만 지정하는 방식입니다.  
스키마 강제는 안 되지만 **LLM이 반드시 JSON을 출력하도록** 강제합니다.  
OpenRouter 일부 모델에서 지원됩니다.

In [ ]:
# 방법 C: json_object mode

try:
    response = llm.invoke(
        [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": TEST_QUERY},
        ],
        max_tokens=2048,
        response_format={"type": "json_object"},
    )
    
    raw = response.content.strip()
    print(f"[LLM 응답]\n{raw}\n")
    
    data_c = json.loads(raw)
    decision_c = RouteResponse(**data_c)
    print(f"✅ 성공: next={decision_c.next}, lotcd={decision_c.lotcd}")
except Exception as e:
    print(f"❌ 실패: {type(e).__name__}: {e}")
    print("   → json_object mode도 이 모델/프록시에서 지원되지 않을 수 있습니다.")

---
## json_repair Edge Case 테스트

LLM이 실제로 생성하는 다양한 malformed JSON을 `json_repair`가 복구하는지 테스트합니다.  
**이 셀은 LLM 호출 없이 실행 가능합니다.**

In [ ]:
cases = [
    (
        "trailing comma",
        '{"next": "yield_agent", "lotcd": "4SS", "filter_params": ["VTH"],}',
        "LLM이 리스트/객체 마지막에 콤마를 붙이는 경우",
    ),
    (
        "single quotes",
        "{'next': 'yield_agent', 'lotcd': '4SS'}",
        "Python dict 스타일로 출력한 경우",
    ),
    (
        "unquoted keys",
        '{next: "yield_agent", lotcd: "4SS"}',
        "JavaScript 객체 스타일로 출력한 경우",
    ),
    (
        "truncated JSON",
        '{"next": "yield_agent", "lotcd": "4SS", "message": "조회하겠',
        "max_tokens 초과로 응답이 잘린 경우",
    ),
    (
        "surrounding text",
        'Here is the result: {"next": "yield_agent", "lotcd": "4SS"} hope this helps',
        "JSON 앞뒤에 설명 텍스트가 붙은 경우",
    ),
    (
        "markdown wrapped",
        '```json\n{"next": "yield_agent", "lotcd": "4SS"}\n```',
        "마크다운 코드블록으로 감싼 경우",
    ),
    (
        "think + JSON",
        '<think>수율 요청이니까 yield로</think>{"next": "yield_agent", "lotcd": "4SS"}',
        "<think> 태그와 JSON이 붙어있는 경우",
    ),
]

print(f"{'케이스':<20} {'결과':<8} {'next':<18} {'lotcd':<6} 설명")
print("─" * 80)

for name, bad_json, desc in cases:
    try:
        # supervisor_node와 동일한 파이프라인
        # 1) <think> 제거
        cleaned = re.sub(r"<think>.*?</think>", "", bad_json, flags=re.DOTALL).strip()
        # 2) 정규식 JSON 추출
        m = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", cleaned, re.DOTALL)
        if not m:
            m = re.search(r"(\{.*\})", cleaned, re.DOTALL)
        if not m:
            m = re.search(r"(\{.*\})", bad_json, re.DOTALL)
        extracted = m.group(1) if m else cleaned
        # 3) json_repair + Pydantic
        repaired = repair_json(extracted)
        data = json.loads(repaired)
        decision = RouteResponse(**data)
        print(f"{name:<20} {'✅':<8} {decision.next:<18} {decision.lotcd:<6} {desc}")
    except Exception as e:
        print(f"{name:<20} {'❌':<8} {'':<18} {'':<6} {desc} → {e}")

---
## 전체 파이프라인 비교 요약

```
방법 A: with_structured_output()
────────────────────────────────
LangChain → API에 response_format.json_schema 전달 → LLM 강제 JSON 출력
                                                      ↑
                                            OpenRouter가 여기서 실패

방법 B: Raw JSON 파싱 (현재 supervisor_node)
────────────────────────────────────────────
프롬프트 "JSON만 출력해" → stream() → <think> 분리
                                        ↓
                                   raw_text
                                        ↓
                              regex로 JSON 추출
                                        ↓
                              json_repair (교정)
                                        ↓
                              json.loads (파싱)
                                        ↓
                            RouteResponse(**data)
                              (Pydantic 검증)
                                        ↓
                           Command(update=..., goto=...)

방법 C: json_object mode
────────────────────────
API에 response_format={"type": "json_object"} 전달
→ LLM이 JSON 출력은 보장하지만 스키마는 강제 안 됨
→ 클라이언트에서 RouteResponse(**data)로 검증 필요
→ OpenRouter + gpt-oss-120b 지원 여부 불확실
```